# Rerun Only the Failures

In this video, I'm going to compute a color for each of 1,673 photos with Pixeltable, then rerun only the photos that failed. In a real job that step is a model call, and you don't want to pay for every photo again because a few failed.

We heard about this from people running big image jobs. One asked whether a run that stops partway loses the whole batch, and another called rerunning only the failed photos "massive."

First, we'll stop a run partway and pick it back up. Then we'll give each of three kinds of failure its own fix.

## Start from an empty `rerun_demo` directory

Start Jupyter with `uv run jupyter lab` in this folder. Nothing here calls a paid API, so you don't need any keys.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
import pixeltable.functions as pxtf

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('rerun_demo', force=True, if_not_exists='ignore')

## The photos table computes two columns from each photo

In `photos`, the only columns you fill in are `name` and `image`. `avg_color` and `color` are computed columns: Pixeltable computes `avg_color` from the image, then `color` from `avg_color`, on every insert.

In [ ]:
!sed -n '5,15p' schema.py

`average_color` stands in for a model call. It's the one demo-only part: it times out the first time it sees about 1 photo in 50, the way a busy API does.

In [ ]:
!sed -n '11,31p' udfs.py

`pxt schema diff` shows the plan, and `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py rerun_demo

In [ ]:
!pxt schema update schema.py rerun_demo -f

## By default, an insert saves all of its rows or none

`photo_names.txt` lists 1,673 small ImageNet photos from the Pixeltable repo. Each `image` is a URL, and Pixeltable downloads the photo when it inserts the row. I'll insert the first 200 the default way.

In [ ]:
BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/imagenette2-160/'
names = open('photo_names.txt').read().split()
rows = [{'name': name, 'image': BASE + name} for name in names]

photos = pxt.get_table('rerun_demo.photos')
try:
    photos.insert(rows[:200])
except pxt.Error as e:
    print(e)
photos.count()

One photo hit a TypeError, so the insert saved none of the 200. That's the default, `on_error='abort'`, and it means a table never holds half a batch.

## Chunks keep what finished when the run stops

For a big run, we change two things. `on_error='ignore'` keeps going past a failed photo and saves its error on the row. The second is chunks: with 200 photos at a time, Pixeltable saves each chunk's results and errors as soon as it finishes.

I'll stop this cell partway through, the way a crash or a closed laptop would.

In [ ]:
def insert_remaining(rows, chunk_size=200):
    done = {row['name'] for row in photos.select(photos.name).collect()}
    todo = [row for row in rows if row['name'] not in done]
    for start in range(0, len(todo), chunk_size):
        photos.insert(todo[start : start + chunk_size], on_error='ignore')


insert_remaining(rows)

*Stop the cell after three or four chunks.*

Pixeltable kept the chunks that finished. Only the chunk in progress is gone.

In [ ]:
photos.count()

I'll run it again. `insert_remaining` skips the photos already in the table, so it picks up where it stopped.

In [ ]:
insert_remaining(rows)
photos.count()

All 1,673 photos are in. Here are a few that worked. `color` names whichever channel of the average is highest.

In [ ]:
photos.where(photos.avg_color.errortype == None).select(
    photos.name, photos.image, photos.avg_color, photos.color
).limit(4).collect()

## Each failure stays on its row, with its error

A photo that failed is still a row. `avg_color.errortype` holds the exception's name, and you can group and filter on it like any column.

In [ ]:
photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

`pxt errors` lists the same failures from the command line, with each message. `color` failed on the same photos, because it's built on `avg_color`.

In [ ]:
!pxt errors rerun_demo/photos 2>/dev/null | head -6

## A retry fixes the timeouts

Timeouts are the failures a retry can fix. `errors_only=True` reruns `avg_color` only on the rows where it failed, then the columns built on it.

In [ ]:
photos.recompute_columns('avg_color', errors_only=True).num_rows

That's how many rows it reran: only the failures, not all 1,673. The stand-in times out only once, so the timeouts are gone now.

In [ ]:
photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

The TypeErrors failed again, because a retry runs the same code on the same photos.

## A code fix, then a retry, covers the grayscale photos

The TypeError means `average_color` expected three color channels. Mode `L` means grayscale, with one channel, and 28 photos are like that.

In [ ]:
photos.where(photos.avg_color.errortype == 'TypeError').select(
    photos.name, photos.image, photos.image.mode, photos.avg_color.errormsg
).limit(3).collect()

The fix is one line in the schema: convert each photo to RGB before `average_color` runs.

In [ ]:
!diff schema.py schema_rgb.py

In [ ]:
!pxt schema diff schema_rgb.py rerun_demo

In [ ]:
!pxt schema update schema_rgb.py rerun_demo -f

The update keeps the colors we already computed. Converting an RGB photo to RGB doesn't change it, so there's no reason to compute those again. The same retry reruns only the failures.

In [ ]:
photos = pxt.get_table('rerun_demo.photos')  # reload after the schema change
photos.recompute_columns('avg_color', errors_only=True)

photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

The grayscale photos have colors now. One failure is left.

## A file that isn't an image gets deleted

`bad_image.JPEG` is 29 bytes of text with a `.JPEG` name. Pixeltable couldn't load it as an image, so `image` holds the error, and `average_color` never ran.

In [ ]:
photos.where(photos.avg_color.errortype != None).select(
    photos.name, photos.image.errortype, photos.avg_color.errortype
).collect()

A retry or a code fix won't help here, so we delete the row.

In [ ]:
photos.delete(where=photos.name == 'bad_image.JPEG')

photos.count(), photos.where(photos.avg_color.errortype != None).count()

## Each kind of failure got one fix

We ran 1,673 photos, and the chunked run kept going past every failure.

| Failure | Photos | Fix |
|---|---|---|
| Timeout, from the stand-in | about 1 in 50 | a retry with `errors_only=True` |
| Grayscale photo | 28 | a code fix, then a retry with `errors_only=True` |
| Not an image | 1 | delete the row |

When the run itself stopped, the chunks that finished stayed saved, and the next run picked up from there.

The notebook and schema files are in the `rerun-failures` folder of the repo linked below. If this helps with your own runs, a star on the Pixeltable GitHub repo helps other developers find it.